<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Weight%20Initialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weight Initialization

Ebben a notebookban a **súly inicializálás** módszereit vizsgáljuk.

## Tartalomjegyzék

1. Inicializálás fontossága
2. Xavier/Glorot inicializálás
3. He/Kaiming inicializálás
4. Inicializálás aktivációs függvények szerint
5. PyTorch inicializálás

## 1. Inicializálás fontossága

### Rossz inicializálás problémái

| Probléma | Ok | Hatás |
|----------|----|-----------|
| Vanishing gradients | Túl kis súlyok | Nincs tanulás |
| Exploding gradients | Túl nagy súlyok | NaN, instabil |
| Szimmetria | Azonos súlyok | Neuronok nem differenciálódnak |

### Cél

Aktivációk és gradiensek varianciájának megőrzése a rétegeken keresztül.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.init as init

np.random.seed(42)
torch.manual_seed(42)

# Rossz inicializálás demonstráció
def forward_pass_numpy(x, weights, activation='tanh'):
    """Forward pass numpy-val."""
    activations = [x]
    h = x

    for w in weights:
        h = h @ w
        if activation == 'tanh':
            h = np.tanh(h)
        elif activation == 'relu':
            h = np.maximum(0, h)
        activations.append(h)

    return activations

# Hálózat: 512 -> 512 -> ... -> 512 (10 réteg)
n_layers = 10
hidden_dim = 512
batch_size = 256

# Input
x = np.random.randn(batch_size, hidden_dim)

# Különböző inicializálások
inits = {
    'Túl kicsi (0.01)': 0.01,
    'Túl nagy (1.0)': 1.0,
    'Standard Normal': 1.0 / np.sqrt(hidden_dim),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, scale) in zip(axes, inits.items()):
    np.random.seed(42)

    if name == 'Standard Normal':
        weights = [np.random.randn(hidden_dim, hidden_dim) * scale for _ in range(n_layers)]
    else:
        weights = [np.random.randn(hidden_dim, hidden_dim) * scale for _ in range(n_layers)]

    activations = forward_pass_numpy(x, weights, activation='tanh')

    # Aktivációk eloszlása rétegenként
    means = [a.mean() for a in activations]
    stds = [a.std() for a in activations]

    ax.errorbar(range(len(activations)), means, yerr=stds, fmt='o-', capsize=5)
    ax.set_xlabel('Réteg')
    ax.set_ylabel('Aktiváció')
    ax.set_title(f'{name}\nVégső std: {stds[-1]:.6f}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Inicializálás hatása az aktivációkra (tanh)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Aktiváció eloszlás hisztogram

fig, axes = plt.subplots(3, n_layers+1, figsize=(20, 10))

for row, (name, scale) in enumerate(inits.items()):
    np.random.seed(42)

    if name == 'Standard Normal':
        weights = [np.random.randn(hidden_dim, hidden_dim) * scale for _ in range(n_layers)]
    else:
        weights = [np.random.randn(hidden_dim, hidden_dim) * scale for _ in range(n_layers)]

    activations = forward_pass_numpy(x, weights, activation='tanh')

    for col, act in enumerate(activations):
        axes[row, col].hist(act.flatten(), bins=50, density=True, alpha=0.7)
        axes[row, col].set_xlim(-2, 2)
        if row == 0:
            axes[row, col].set_title(f'Réteg {col}')
        if col == 0:
            axes[row, col].set_ylabel(name[:15])

plt.suptitle('Aktiváció eloszlás rétegenként', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Xavier/Glorot inicializálás

### Xavier uniform

$$W \sim U\left[-\frac{\sqrt{6}}{\sqrt{n_{in} + n_{out}}}, \frac{\sqrt{6}}{\sqrt{n_{in} + n_{out}}}\right]$$

### Xavier normal

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{in} + n_{out}}\right)$$

### Mikor használd

- Tanh aktiváció
- Sigmoid aktiváció
- Lineáris rétegek

In [ ]:
# Xavier inicializálás implementáció

def xavier_uniform(fan_in, fan_out):
    limit = np.sqrt(6 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

def xavier_normal(fan_in, fan_out):
    std = np.sqrt(2 / (fan_in + fan_out))
    return np.random.randn(fan_in, fan_out) * std

# Összehasonlítás
np.random.seed(42)
weights_xavier_u = [xavier_uniform(hidden_dim, hidden_dim) for _ in range(n_layers)]

np.random.seed(42)
weights_xavier_n = [xavier_normal(hidden_dim, hidden_dim) for _ in range(n_layers)]

act_xavier_u = forward_pass_numpy(x, weights_xavier_u, activation='tanh')
act_xavier_n = forward_pass_numpy(x, weights_xavier_n, activation='tanh')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Xavier uniform
stds_u = [a.std() for a in act_xavier_u]
axes[0].plot(stds_u, 'bo-', linewidth=2, markersize=10)
axes[0].axhline(y=1/np.sqrt(3), color='r', linestyle='--', label=f'Tanh szaturáció előtt')
axes[0].set_xlabel('Réteg')
axes[0].set_ylabel('Aktiváció std')
axes[0].set_title('Xavier Uniform')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Xavier normal
stds_n = [a.std() for a in act_xavier_n]
axes[1].plot(stds_n, 'go-', linewidth=2, markersize=10)
axes[1].axhline(y=1/np.sqrt(3), color='r', linestyle='--', label=f'Tanh szaturáció előtt')
axes[1].set_xlabel('Réteg')
axes[1].set_ylabel('Aktiváció std')
axes[1].set_title('Xavier Normal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Xavier inicializálás tanh aktivációval', fontsize=14)
plt.tight_layout()
plt.show()

## 3. He/Kaiming inicializálás

### He uniform

$$W \sim U\left[-\sqrt{\frac{6}{n_{in}}}, \sqrt{\frac{6}{n_{in}}}\right]$$

### He normal

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{in}}\right)$$

### Mikor használd

- ReLU aktiváció
- LeakyReLU, PReLU
- Modern CNN-ek

In [ ]:
# He inicializálás

def he_uniform(fan_in, fan_out):
    limit = np.sqrt(6 / fan_in)
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

def he_normal(fan_in, fan_out):
    std = np.sqrt(2 / fan_in)
    return np.random.randn(fan_in, fan_out) * std

# ReLU-val tesztelés
np.random.seed(42)
weights_xavier_relu = [xavier_normal(hidden_dim, hidden_dim) for _ in range(n_layers)]

np.random.seed(42)
weights_he = [he_normal(hidden_dim, hidden_dim) for _ in range(n_layers)]

act_xavier_relu = forward_pass_numpy(x, weights_xavier_relu, activation='relu')
act_he_relu = forward_pass_numpy(x, weights_he, activation='relu')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Xavier + ReLU (rossz)
stds_x = [a.std() for a in act_xavier_relu]
axes[0].plot(stds_x, 'ro-', linewidth=2, markersize=10)
axes[0].set_xlabel('Réteg')
axes[0].set_ylabel('Aktiváció std')
axes[0].set_title(f'Xavier + ReLU\nVégső std: {stds_x[-1]:.4f}')
axes[0].grid(True, alpha=0.3)

# He + ReLU (jó)
stds_h = [a.std() for a in act_he_relu]
axes[1].plot(stds_h, 'go-', linewidth=2, markersize=10)
axes[1].set_xlabel('Réteg')
axes[1].set_ylabel('Aktiváció std')
axes[1].set_title(f'He + ReLU\nVégső std: {stds_h[-1]:.4f}')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Xavier vs He inicializálás ReLU aktivációval', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Inicializálás aktivációs függvények szerint

### Összefoglaló táblázat

| Aktiváció | Inicializálás | Miért |
|-----------|--------------|-------|
| Sigmoid | Xavier | Szimmetrikus |
| Tanh | Xavier | Szimmetrikus |
| ReLU | He | Csak pozitív |
| LeakyReLU | He (módosított) | Majdnem ReLU |
| SELU | LeCun | Self-normalizing |

In [ ]:
# Összehasonlítás különböző aktivációkkal

def lecun_normal(fan_in, fan_out):
    std = np.sqrt(1 / fan_in)
    return np.random.randn(fan_in, fan_out) * std

# SELU aktiváció
def selu(x):
    alpha = 1.6732632423543772848170429916717
    scale = 1.0507009873554804934193349852946
    return scale * np.where(x > 0, x, alpha * (np.exp(x) - 1))

# Leaky ReLU
def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

activations_dict = {
    'tanh': np.tanh,
    'relu': lambda x: np.maximum(0, x),
    'leaky_relu': leaky_relu,
    'selu': selu
}

init_dict = {
    'tanh': xavier_normal,
    'relu': he_normal,
    'leaky_relu': he_normal,
    'selu': lecun_normal
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (act_name, act_fn) in zip(axes, activations_dict.items()):
    init_fn = init_dict[act_name]

    np.random.seed(42)
    weights = [init_fn(hidden_dim, hidden_dim) for _ in range(n_layers)]

    # Custom forward pass
    activations = [x]
    h = x
    for w in weights:
        h = h @ w
        h = act_fn(h)
        activations.append(h)

    stds = [a.std() for a in activations]

    ax.plot(stds, 'o-', linewidth=2, markersize=10)
    ax.set_xlabel('Réteg')
    ax.set_ylabel('Aktiváció std')
    ax.set_title(f'{act_name.upper()} + {init_fn.__name__}\nVégső std: {stds[-1]:.4f}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Ajánlott inicializálás aktivációnként', fontsize=14)
plt.tight_layout()
plt.show()

## 5. PyTorch inicializálás

### PyTorch init modul

In [ ]:
# PyTorch inicializálási függvények

class Net(nn.Module):
    def __init__(self, init_type='default'):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        # Inicializálás
        self._init_weights(init_type)

    def _init_weights(self, init_type):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init_type == 'xavier_uniform':
                    init.xavier_uniform_(m.weight)
                elif init_type == 'xavier_normal':
                    init.xavier_normal_(m.weight)
                elif init_type == 'kaiming_uniform':
                    init.kaiming_uniform_(m.weight, nonlinearity='relu')
                elif init_type == 'kaiming_normal':
                    init.kaiming_normal_(m.weight, nonlinearity='relu')
                elif init_type == 'zeros':
                    init.zeros_(m.weight)
                elif init_type == 'ones':
                    init.ones_(m.weight)
                elif init_type == 'orthogonal':
                    init.orthogonal_(m.weight)

                if m.bias is not None:
                    init.zeros_(m.bias)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

# Súly eloszlások összehasonlítása
init_types = ['xavier_uniform', 'xavier_normal', 'kaiming_uniform', 'kaiming_normal', 'orthogonal']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for ax, init_type in zip(axes, init_types):
    torch.manual_seed(42)
    model = Net(init_type)

    # Első réteg súlyai
    weights = model.fc1.weight.data.numpy().flatten()

    ax.hist(weights, bins=50, density=True, alpha=0.7)
    ax.set_title(f'{init_type}\nmean={weights.mean():.4f}, std={weights.std():.4f}')
    ax.set_xlabel('Súly értéke')
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')

plt.suptitle('PyTorch inicializálási módszerek', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Inicializálás hatása tanításra

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)

def train_with_init(init_type, epochs=100):
    torch.manual_seed(42)
    model = Net(init_type)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return losses

# Összehasonlítás
plt.figure(figsize=(12, 6))

for init_type in ['xavier_uniform', 'kaiming_uniform', 'orthogonal', 'zeros']:
    losses = train_with_init(init_type)
    plt.plot(losses, label=init_type, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Inicializálás hatása a tanulásra')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Összefoglalás

### Inicializálási módszerek:

| Módszer | Képlet | Használat |
|---------|--------|----------|
| Xavier | $\sqrt{\frac{2}{n_{in}+n_{out}}}$ | Tanh, Sigmoid |
| He | $\sqrt{\frac{2}{n_{in}}}$ | ReLU, LeakyReLU |
| LeCun | $\sqrt{\frac{1}{n_{in}}}$ | SELU |

### PyTorch használat:

```python
# Xavier
nn.init.xavier_uniform_(layer.weight)
nn.init.xavier_normal_(layer.weight)

# He/Kaiming
nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
```

### Best practices:

1. ReLU → He inicializálás
2. Tanh/Sigmoid → Xavier inicializálás
3. Bias-t nullára inicializáld